# 라벨 분포 그림과 원본 캡션·object-only 임베딩
Drive의 `fmri_capstone/colab_bundle/`에 `captions.jsonl.gz`, `labels_pass0.jsonl.gz`가 있어야 한다.
NSD 데이터 사용 약관에 따라 이 Drive 폴더는 다른 사람과 공유하지 않는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fmri_capstone'
!pip -q install sentence-transformers matplotlib

In [ ]:
import gzip, json, numpy as np, collections
B = BASE + '/colab_bundle'
caps = [json.loads(l) for l in gzip.open(B + '/captions.jsonl.gz', 'rt')]
labs = {r['cocoId']: r for r in map(json.loads, gzip.open(B + '/labels_pass0.jsonl.gz', 'rt'))}
print(len(caps), len(labs))

## 1. 이미지당 라벨 수 분포 (코어 v9 / 그 밖 v10)

In [ ]:
import matplotlib.pyplot as plt
DIMS = ['place_type','environment','activity','implied_event','scale','notes']
fig, axes = plt.subplots(2, 6, figsize=(20, 6), sharey='row')
for row, (ver, title) in enumerate([('v9', 'core (v9)'), ('v10', 'rest (v10)')]):
    sub = [r for r in labs.values() if r['prompt_version'] == ver]
    for j, d in enumerate(DIMS):
        c = collections.Counter(min(len(r['labels'][d]), 5) for r in sub)
        axes[row, j].bar(range(6), [c.get(k, 0) / len(sub) for k in range(6)])
        axes[row, j].set_title(f'{title} {d}'); axes[row, j].set_xticks(range(6)); axes[row, j].set_xticklabels(['0','1','2','3','4','5+'])
plt.tight_layout(); plt.savefig(BASE + '/label_count_hist.png', dpi=120); plt.show()

## 2. 값 빈도 분포 (로그-로그, 흔한 값부터)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(22, 3.5))
for j, d in enumerate(DIMS):
    c = collections.Counter(v[0].casefold().strip() for r in labs.values() for v in r['labels'][d])
    freq = sorted(c.values(), reverse=True)
    axes[j].loglog(range(1, len(freq)+1), freq); axes[j].set_title(f'{d} (고유 {len(c):,})')
plt.tight_layout(); plt.savefig(BASE + '/value_freq.png', dpi=120); plt.show()

## 2-2. 라벨 값의 의미 군집 (MPNet으로 고유 값을 임베딩)
문자열 빈도가 아니라 의미 공간에서 라벨이 얼마나 퍼져 있는지 본다. 진단용이며 합격선은 두지 않는다.

In [ ]:
import torch, itertools
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
norm = lambda v: v.casefold().strip()
cnt = {d: collections.Counter(norm(v[0]) for r in labs.values() for v in r['labels'][d]) for d in DIMS}
vals = {d: sorted(cnt[d]) for d in DIMS}
idx = {d: {v: i for i, v in enumerate(vals[d])} for d in DIMS}
st = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=dev)
VE = {d: st.encode(vals[d], batch_size=256, normalize_embeddings=True, show_progress_bar=False) for d in DIMS}
print({d: len(vals[d]) for d in DIMS})

In [ ]:
# (1) 군집 크기 분포: 라벨 출현 수로 가중. 문자열 기준 쏠림(가장 흔한 값 비율)과 비교
for d in DIMS:
    k = min(100, max(2, len(vals[d]) // 5))
    km = KMeans(k, n_init=3, random_state=0).fit(VE[d])
    w = np.array([cnt[d][v] for v in vals[d]], dtype=float)
    share = np.bincount(km.labels_, weights=w, minlength=k); share /= share.sum()
    eff = np.exp(-(share[share > 0] * np.log(share[share > 0])).sum())
    top = np.argsort(-share)[:3]
    ex = [vals[d][int(np.argmax((km.labels_ == c) * (VE[d] @ km.cluster_centers_[c])))] for c in top]
    print(f'{d:14s} 고유 {len(vals[d]):6d} | 군집 {k} | 최대 군집 {share[top[0]]:.1%}, 상위 3개 {share[top].sum():.1%}, 유효 군집 수 {eff:.1f} | 대표: {ex}')

In [ ]:
# (2) 한 이미지 안의 라벨 쌍은 서로 다른 해석인가, 바꿔쓰기인가 (임의의 라벨 쌍과 비교)
rng = np.random.default_rng(0)
for d in ['place_type', 'environment', 'activity', 'scale']:
    within = []
    for r in labs.values():
        ii = [idx[d][norm(v[0])] for v in r['labels'][d]]
        if len(ii) >= 2:
            within += [float(VE[d][a] @ VE[d][b]) for a, b in itertools.combinations(ii, 2)]
    occ = np.array([idx[d][norm(v[0])] for r in labs.values() for v in r['labels'][d]])
    pr = rng.choice(occ, (len(within), 2))
    base = (VE[d][pr[:, 0]] * VE[d][pr[:, 1]]).sum(1)
    within = np.array(within)
    print(f'{d:12s} 같은 이미지 안 평균 코사인 {within.mean():.3f} | 임의 쌍 {base.mean():.3f} | 0.8 이상(바꿔쓰기에 가까움) {np.mean(within > 0.8):.1%}')

In [ ]:
# (3) 항목 사이 겹침: 항목별 평균 임베딩(출현 가중)의 코사인
cent = {}
for d in DIMS:
    w = np.array([cnt[d][v] for v in vals[d]], dtype=float)
    c = (VE[d] * w[:, None]).sum(0); cent[d] = c / np.linalg.norm(c)
import pandas as pd
print(pd.DataFrame([[float(cent[a] @ cent[b]) for b in DIMS] for a in DIMS], index=DIMS, columns=DIMS).round(2))
# (3-2) 코어(v9)와 그 밖(v10)의 항목별 평균 임베딩 거리
for d in DIMS:
    m = {}
    for ver in ('v9', 'v10'):
        vs = [norm(v[0]) for r in labs.values() if r['prompt_version'] == ver for v in r['labels'][d]]
        e = VE[d][[idx[d][v] for v in vs]].mean(0); m[ver] = e / np.linalg.norm(e)
    print(f'{d:14s} v9-v10 평균 코사인 {float(m["v9"] @ m["v10"]):.3f}')

In [ ]:
# (4) scale이 물리적 크기와 구도 두 덩어리로 갈리는지: 2~6개 군집의 대표 값
km = KMeans(6, n_init=5, random_state=0).fit(VE['scale'])
for c in range(6):
    mem = np.where(km.labels_ == c)[0]
    top = mem[np.argsort(-(VE['scale'][mem] @ km.cluster_centers_[c]))[:6]]
    print(c, len(mem), [vals['scale'][i] for i in top])

## 2-3. 보고서용 그림 (라벨 군집 지도, 한 이미지 안 라벨 간 유사도)
영문 제목을 써서 한글 폰트 문제를 피한다. 그림은 `figures/`에 저장된다.

In [ ]:
import os
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt, textwrap
os.makedirs(BASE + '/figures', exist_ok=True)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 3, figsize=(21, 13))
for ax, d in zip(axes.ravel(), DIMS):
    n = len(vals[d]); sel = np.sort(rng.choice(n, min(n, 6000), replace=False))
    X = VE[d][sel]
    k = min(60, max(2, len(sel) // 10))
    km = KMeans(k, n_init=3, random_state=0).fit(X)
    w = np.array([cnt[d][vals[d][i]] for i in sel], dtype=float)
    share = np.bincount(km.labels_, weights=w, minlength=k); share /= share.sum()
    order = np.argsort(-share)[:8]
    xy = TSNE(2, init='pca', perplexity=30, random_state=0).fit_transform(X)
    ax.scatter(xy[:, 0], xy[:, 1], s=4, c='#cccccc')
    cmap = plt.get_cmap('tab10')
    for r, c in enumerate(order):
        m = km.labels_ == c
        ax.scatter(xy[m, 0], xy[m, 1], s=6, color=cmap(r))
        rep = np.where(m)[0][np.argmax(X[m] @ km.cluster_centers_[c])]
        lab = textwrap.shorten(vals[d][sel[rep]], 44, placeholder='…')
        ax.annotate(f'{lab} ({share[c]:.1%})', xy[m].mean(0), xytext=(0, 16 if r % 2 == 0 else -16), textcoords='offset points',
                    ha='center', fontsize=8.5, weight='bold', bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=cmap(r), alpha=0.9),
                    arrowprops=dict(arrowstyle='-', color=cmap(r), lw=0.8))
    ax.set_title(f'{d}: {len(vals[d]):,} unique values, 8 largest of {k} clusters labeled', fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(BASE + '/figures/label_cluster_map.png', dpi=160); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8), sharey=True)
for ax, d in zip(axes, ['place_type', 'environment', 'activity', 'scale']):
    within = []
    for r in labs.values():
        ii = [idx[d][norm(v[0])] for v in r['labels'][d]]
        if len(ii) >= 2: within += [float(VE[d][a] @ VE[d][b]) for a, b in itertools.combinations(ii, 2)]
    occ = np.array([idx[d][norm(v[0])] for r in labs.values() for v in r['labels'][d]])
    pr = rng.choice(occ, (len(within), 2)); base = (VE[d][pr[:, 0]] * VE[d][pr[:, 1]]).sum(1)
    bins = np.linspace(0, 1, 41)
    ax.hist(base, bins, alpha=.5, density=True, label='random label pairs'); ax.hist(within, bins, alpha=.5, density=True, label='labels of the same image')
    ax.set_title(d); ax.set_xlabel('cosine similarity')
axes[0].legend(); plt.tight_layout(); plt.savefig(BASE + '/figures/within_image_similarity.png', dpi=160); plt.show()

## 3. 임베딩 (원본 캡션, object-only)
- 원본 캡션: 이미지당 5개를 각각 임베딩해 평균(길이 제한 때문에 이어 붙이지 않는다).
- object-only: COCO 사물 범주명을 알파벳순으로 쉼표 나열한 문장.
- 모델: CLIP ViT-L/14 텍스트(77토큰 제한), all-mpnet-base-v2, Qwen3-Embedding-0.6B, gtr-t5-base.
- 결과는 `embeddings/{모델}_{텍스트}.npy`(float16)로 저장하고, 행 순서는 `embeddings/ids.json`의 cocoId 순서다.
- context-only 텍스트는 만드는 방식이 정해지기 전이라 이 노트북에서 만들지 않는다.

In [ ]:
import os, torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPTokenizer
os.makedirs(BASE + '/embeddings', exist_ok=True)
ids = [r['cocoId'] for r in caps]
json.dump(ids, open(BASE + '/embeddings/ids.json', 'w'))
flat_caps = [(i, c) for i, r in enumerate(caps) for c in r['captions']]
obj_text = [r['object_only'] if r['object_only'] else 'no object' for r in caps]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def mean_per_image(E, owner):
    out = np.zeros((len(caps), E.shape[1]), dtype=np.float32); n = np.zeros(len(caps))
    for e, o in zip(E, owner): out[o] += e; n[o] += 1
    out /= np.maximum(n, 1)[:, None]
    return out / np.linalg.norm(out, axis=1, keepdims=True)

def save(name, kind, E): np.save(f'{BASE}/embeddings/{name}_{kind}.npy', E.astype(np.float16))

def run_st(name, model_id, **kw):
    m = SentenceTransformer(model_id, device=dev, **kw)
    E = m.encode([c for _, c in flat_caps], batch_size=256, normalize_embeddings=True, show_progress_bar=True)
    save(name, 'caption', mean_per_image(E, [i for i, _ in flat_caps]))
    save(name, 'object', m.encode(obj_text, batch_size=256, normalize_embeddings=True))
    del m; torch.cuda.empty_cache()

def run_clip(name='clip_vit_l14', model_id='openai/clip-vit-large-patch14'):
    tok = CLIPTokenizer.from_pretrained(model_id); m = CLIPModel.from_pretrained(model_id).to(dev).half().eval()
    def enc(texts):
        out = []
        for k in range(0, len(texts), 256):
            t = tok(texts[k:k+256], padding=True, truncation=True, max_length=77, return_tensors='pt').to(dev)
            with torch.no_grad(): e = m.text_projection(m.text_model(**t).pooler_output)
            out.append(torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy())
        return np.concatenate(out)
    E = enc([c for _, c in flat_caps])
    save(name, 'caption', mean_per_image(E, [i for i, _ in flat_caps])); save(name, 'object', enc(obj_text))
    over = sum(len(tok(c)['input_ids']) > 77 for _, c in flat_caps)
    print('CLIP 77토큰 초과 캡션 수:', over)

run_clip()
run_st('mpnet_base_v2', 'sentence-transformers/all-mpnet-base-v2')
run_st('qwen3_emb_0p6b', 'Qwen/Qwen3-Embedding-0.6B')
run_st('gtr_t5_base', 'sentence-transformers/gtr-t5-base')
print(os.listdir(BASE + '/embeddings'))

## 4. 이미지 임베딩 (CLIP ViT-L/14 image, 기준선 사다리의 CLIP-image)
- `stim_info.jsonl.gz`(cocoId, cocoSplit, cropBox)도 `colab_bundle/`에 올려 둬야 한다.
- 이미지는 COCO 공개 서버에서 받는다. NSD 피험자가 본 것처럼 cropBox(top, bottom, left, right 비율)로 잘라서 임베딩한다.
- 결과: `embeddings/clip_vit_l14_image.npy`, 행 순서는 `ids.json`과 같다.

In [ ]:
import ast, io, time, requests
import concurrent.futures as cf
from PIL import Image
from transformers import CLIPModel, CLIPImageProcessor
stim = {r['cocoId']: r for r in map(json.loads, gzip.open(B + '/stim_info.jsonl.gz', 'rt'))}
ids = json.load(open(BASE + '/embeddings/ids.json'))
mid = 'openai/clip-vit-large-patch14'
proc = CLIPImageProcessor.from_pretrained(mid)
cm = CLIPModel.from_pretrained(mid).to(dev).half().eval()

def load_img(cid):
    s = stim[cid]
    url = f"http://images.cocodataset.org/{s['cocoSplit']}/{int(cid):012d}.jpg"
    for k in range(4):
        try:
            im = Image.open(io.BytesIO(requests.get(url, timeout=30).content)).convert('RGB'); break
        except Exception:
            time.sleep(1 + k)
    else:
        raise RuntimeError(url)
    top, bot, left, right = ast.literal_eval(s['cropBox'])
    w, h = im.size
    return im.crop((round(left * w), round(top * h), round(w - right * w), round(h - bot * h)))

out = []
with cf.ThreadPoolExecutor(32) as ex:
    for k in range(0, len(ids), 256):
        ims = list(ex.map(load_img, ids[k:k + 256]))
        px = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev).half()
        with torch.no_grad():
            e = cm.visual_projection(cm.vision_model(pixel_values=px).pooler_output)
        out.append(torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy())
        if (k // 256) % 20 == 0: print(k, len(ids), flush=True)
np.save(BASE + '/embeddings/clip_vit_l14_image.npy', np.concatenate(out).astype(np.float16))
print('저장 완료', np.concatenate(out).shape)